In [ ]:
import numpy as np
import mne
import os

import math

import matplotlib.pyplot as plt
%matplotlib qt

In [ ]:
# load data
# 1. raw
# 2. event : onset, ME, MI, grasp, sequence
# 3. position data : pos, pos_target, vel


mne.set_log_level('WARNING') # no console prints

data_dir = r'./raw_data' # USER_INPUT : where ECoG data folder is saved
epo_dir = "./preprocessed_data" # USER_INPUT : save directory

if not os.path.exists(epo_dir):
    os.makedirs(epo_dir)

# load data

subs = {
    1:  {'name': '',    
         'bads': [], # bad channel name
         'depths': [], # non-ecog channel name
         'session': 0,  # session to use
         'drop_s1': [], # bad trial num
         'drop_s2': [], # bad trial num
         'notch': [], # notch filter (not used)
         'n_session1': 0}, # number of trials in session 1
}


In [ ]:
def save_epoch (sub_num, subs, data_dir, epo_dir, filter, notch_width):
 
    # set parameters
    tmin, tmax = 1, 3 # USER INPUT: for epoching
    sub_name = subs[sub_num]['name']
    session_num = subs[sub_num]['session']
    
    for session in range(1, session_num + 1):

        # load data
        fname = f"{data_dir}/{sub_name}_session_{session}_raw.fif"
        raw = mne.io.read_raw_fif(fname, preload=True)
        
        # load meta data
        fname = f"{data_dir}/{sub_name}_session_{session}_meta.npz"
        meta = np.load(fname)        
    
        for key in meta.keys():
            globals()[key] = meta[key]
        
        # 1. bandpass filter (3th order butterworth)
        # 100Hz downsampling
        raw_filt = raw.copy().filter(0.2, 8., fir_design='firwin', l_trans_bandwidth='auto', h_trans_bandwidth='auto', filter_length='auto', phase='zero')
        
        # # 2. notch filter (1. 60 Hz harmonics, 2. subject dependents): raw --> raw_filt -----------------------------------------------------------------------
        
        # line = np.arange(60, 160, 60) # line noise: 60Hz harmonics
        # notch = np.array(subs[sub_num]["notch"]) # subject dependent noise  
        # freqs = np.concatenate((line, notch))

        # if filter == 'FIR':
        #     # 1-2-1. FIR filter (width = 1 Hz)
        #     raw_filt.notch_filter(freqs, notch_widths = notch_width)
            
        # elif filter == 'IIR':
        #     # 1-2-2. IIR filter
        #     iir_params = {
        #         'ftype': 'butter',  # Butterworth filter
        #         'order': 2,         # Lower order to avoid instability
        #         'output': 'sos'     # Use second-order sections for stability
        #     }
            
        #     for freq in freqs:
        #         raw_filt.notch_filter(freqs=[freq], notch_widths=notch_width, method='iir', iir_params=iir_params) # apply iir filter
                
        # else:
        #     # Error message
        #     print("choose filter FIR or IIR")

        
        # # 3. CAR (ME, MI): pre_epochs --> epochs --------------------------------------------------------------------------------------------------------------
        
        # raw_filt.set_eeg_reference('average')

        
        # 4. event epoching (ME, MI): raw_filt --> pre_epochs ------------------------------------------------------------------------------------------------
        
        event_id = {'ME/1': 11, 'ME/2': 12, 'ME/3': 13, 'ME/4': 14,
                    'MI/1': 21, 'MI/2': 22, 'MI/3': 23, 'MI/4': 24}
        
        epochs = mne.Epochs(raw_filt, events, event_id=event_id, tmin=-tmin, tmax=tmax, baseline=(None, 0))


        # plt save  ============================================================================================================================================

        # psd plot save
        plt_dir =  f"{epo_dir}/plots"
                
        if not os.path.exists(plt_dir):
            os.makedirs(plt_dir)

        pname = f"sub{sub_num}_{sub_name}_preprocessed"

        psd = epochs.compute_psd(fmin=0, fmax=180)

        fig = psd.plot(show = False)

        if hasattr(fig, 'axes'):
            ax = fig.axes[0] 
            ax.set_title(pname) 
            plt.show()
        else:
            print("Unable to access axes of the plot")
        
        # save
        plt.savefig("psd_plot.png", dpi=300)
        plt.close()

        # =====================================================================================================================================================
        
        # 4. epoch positon, velocity data (vel_epo, pos_epo) ---------------------------------------------------------------------------------------------------
        sfreq = raw.info["sfreq"] # 2000 Hz
        
        # pos, vel, ME, MI, sequence
        data_len = len(range(-np.int64(np.round(sfreq*tmin)), np.int64(np.round(sfreq*tmax)) + 1))
        vel_epo = np.zeros([3, data_len, 0])
        pos_epo =  np.zeros([3, data_len, 0])

        # make epoched vel, pos data
        for i in range(len(ME)):
            
            ME_onset = ME[i,0] - 1 # matlab to python (idx-1)
            ME_start = ME_onset - np.int64(sfreq*tmin) # -tmin sec
            ME_end   = ME_onset + np.int64(sfreq*tmax) # +tmax sec
  
            seg_vel = vel[:, ME_start : ME_end + 1]
            vel_epo = np.append(vel_epo, seg_vel.reshape([seg_vel.shape[0], seg_vel.shape[1], 1]), axis = 2)
        
            seg_pos = pos[:, ME_start : ME_end + 1]
            pos_epo = np.append(pos_epo, seg_pos.reshape([seg_pos.shape[0], seg_pos.shape[1], 1]), axis = 2)
                
   
        # 5. Save epoched data ----------------------------------------------------------------------------------------------------------------------------------
        
        # 5-1. epochs (ecog data)
        fname = f"{epo_dir}/{sub_name}_session_{session}_epo.fif"
        epochs.save(fname, overwrite=True)

        # 5-1. pos, vel, sequence (motion capture data)
        fname = f"{epo_dir}/{sub_name}_session_{session}_meta.npz"
        save_dict = {}
        save_dict['sequence'] = sequence
        save_dict['pos'] = pos_epo
        save_dict['vel'] = vel_epo
        
        np.savez_compressed(fname, **save_dict)


## save

In [ ]:
sub_num = 1 # USER_INPUT
filter = 'FIR'
notch_width = 2

sub_name = subs[sub_num]['name']
print(f"------------------{sub_num}: {sub_name} -----------------")

save_epoch(sub_num, subs, data_dir, epo_dir, filter, notch_width)